# TitanAgent 実験2: 小型 Titans (MAC) による思考整理・壁打ち学習（SFT → GRPO）

**Kaggle 無料枠（T4 16GB / P100 16GB）で OOM を防ぎながら完走できる**、「質問力（逆質問）→ 多段深掘り → 長期記憶を活かした壁打ち」の3段階カリキュラム実験ノートブックです。

### 学習する3段階カリキュラム
1. **Level 1（質問力・不足の見抜き）**: 曖昧な相談から欠けている前提（`目的` / `制約` / `優先軸`）を1つ見抜く → `<ask>` で逆質問 → `<user>` の返答を受けて `<summary>` で論点と方針を整理
2. **Level 2（多段深掘り・論点整理）**: 複数の前提が欠けている相談に対し、2回連続で `<ask>` を回して重複なく前提を引き出し、全体を統合して整理
3. **Level 3（長期記憶を活かした壁打ち）**: 前の相談でユーザーが話した前提（`制約` や `優先軸`）とディストラクタの後、新しい相談では**すでに知っている前提を聞き直さず**、記憶から引き出して本当に足りない1点だけを質問・整理

### 実験1.5 からの改善点（GRPO の `skip(zero-var)` 対策）
- SFT をやや余白を残したステップ数（`450`）で止め、GRPO に学習の伸びしろを残す
- 段階的な部分報酬（スロット一致率・的確な逆質問・聞き直しペナルティ・記憶再利用ボーナス）を導入し、グループ内分散を確保
- GRPO は難しめの Level 2 / Level 3（ディストラクタ1〜2件）を重点出題し、評価も各レベル40問で安定比較

In [ ]:
# 1. 環境確認 & ライブラリインストール
import os, sys, torch

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print(f'Environment: Colab={IN_COLAB}, Kaggle={IN_KAGGLE}')
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip -q install -U titans-pytorch

In [ ]:
# 2. 実験設定（Kaggle T4 16GB で OOM しない安全デフォルト）
from pathlib import Path

PRESET = 'tiny'          # 'micro' (超高速確認), 'tiny' (推奨・~10M), 'small' (~22M)
SFT_STEPS = 450          # Stage 1 (SFT): 過学習を避けてGRPOの余白を残す
GRPO_STEPS = 120         # Stage 2 (GRPO): Level 2 & 3 重点強化
GROUP_SIZE = 4           # 1問あたりの生成本数 (4 が 16GB VRAM で安全)
GRPO_TEMP = 0.95         # グループ内多様性を確保する温度
MAX_NEW_TOKENS = 190     # RL生成の最大長
KL_BETA = 0.04           # SFT基礎が崩れないためのKL制約
EVAL_PER_LEVEL = 40      # 各レベルの評価問題数
SEED = 42

if IN_KAGGLE:
    OUTPUT_DIR = Path('/kaggle/working/checkpoints/exp2_thinking_partner')
else:
    OUTPUT_DIR = Path('./checkpoints/exp2_thinking_partner')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRESETS = {
    'micro': dict(dim=128, depth=3, segment_len=32, num_persist_mem_tokens=4, num_longterm_mem_tokens=8,  seq_len=448, sft_batch_size=8, sft_grad_accum=2,  sft_lr=1.5e-3, grpo_lr=1e-4),
    'tiny':  dict(dim=256, depth=4, segment_len=64, num_persist_mem_tokens=4, num_longterm_mem_tokens=16, seq_len=448, sft_batch_size=4, sft_grad_accum=8,  sft_lr=8e-4,   grpo_lr=5e-5),
    'small': dict(dim=384, depth=6, segment_len=64, num_persist_mem_tokens=8, num_longterm_mem_tokens=16, seq_len=448, sft_batch_size=2, sft_grad_accum=16, sft_lr=5e-4,   grpo_lr=3e-5),
}
cfg = PRESETS[PRESET]
print('OUTPUT_DIR =', OUTPUT_DIR)
print('Selected preset:', PRESET, cfg)

In [ ]:
# 3. 自前トークナイザー & 3段階壁打ちカリキュラム生成器（疑似ユーザー環境つき）
import copy, json, math, random, re, time
from contextlib import nullcontext
from dataclasses import dataclass
import torch
import torch.nn.functional as F
from titans_pytorch import MemoryAsContextTransformer

torch.manual_seed(SEED)
random.seed(SEED)

SPECIAL_TOKENS = [
    '<|pad|>', '<|eos|>', '<|unk|>',
    '<think>', '</think>', '<ask>', '</ask>',
    '<user>', '</user>', '<summary>', '</summary>',
]

TOPICS = [
    '副業', 'アプリ開発', '英語学習', '転職',
    'ブログ執筆', '資格勉強', '運動習慣', '読書習慣',
    '個人制作', '研究テーマ', '時間管理', '発信活動',
    '資産形成', 'デザイン学習', '自動化ツール', 'キャリア設計',
]

SLOT_NAMES = ['目的', '制約', '優先軸']

SLOT_VALUES = {
    '目的': ['収益化', 'スキル習得', '習慣化', '実績作り', 'キャリア転換', '好奇心探究', '業務効率化', '自己表現'],
    '制約': ['平日30分', '週末のみ', '予算ゼロ', '1日1時間', '期限1か月', 'スマホ中心', '低負荷', '夜間のみ'],
    '優先軸': ['継続しやすさ', '短期成果', '楽しさ', '質重視', '低リスク', '独学完結'],
}

PRIORITY_TO_PLAN = {
    '継続しやすさ': '最小単位で毎日回す',
    '短期成果': '期限を切り一点集中',
    '楽しさ': '興味軸で小さく試す',
    '質重視': '設計を固めて深掘り',
    '低リスク': '無料枠で検証優先',
    '独学完結': '教材を絞り自走する',
}

CONSULT_TEMPLATES = [
    '相談: {topic}を考えている。{known_text}。どう進めるべき？',
    '相談: {topic}について思考を整理したい。{known_text}。方針を決めたい。',
    '相談: {topic}を始めたいが迷っている。{known_text}。',
]

DISTRACTOR_TURNS = [
    'メモ: 今の考えをそのまま保持中。',
    'メモ: 少し休憩してから次の相談へ移る。',
    'メモ: さっきの前提は今後も活かせそう。',
    'メモ: 別のテーマも続けて壁打ちしたい。',
    'メモ: 頭の中を少しずつ言語化している。',
]

class ThinkingTokenizer:
    def __init__(self, token_to_id):
        self.token_to_id = dict(token_to_id)
        self.id_to_token = {v: k for k, v in self.token_to_id.items()}
        self.pad_token_id = self.token_to_id['<|pad|>']
        self.eos_token_id = self.token_to_id['<|eos|>']
        self.unk_token_id = self.token_to_id['<|unk|>']
        self.ask_end_id = self.token_to_id['</ask>']
        self.vocab_size = len(self.token_to_id)
        self._split_re = re.compile('(' + '|'.join(re.escape(t) for t in SPECIAL_TOKENS) + ')')

    @classmethod
    def build_default(cls):
        chars = set('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz+-*/=|():.,!? \n、。')
        for s in [
            '相談: テーマ: 既知: 記憶: 不足: 確認: 整理完了。全前提が揃った。',
            '方針=エラー: 該当なし', 'テーマ=目的=制約=優先軸=方針=',
            'を聞く。次はが必要。前回の前提を再利用。',
            *TOPICS, *SLOT_NAMES,
            *[v for vals in SLOT_VALUES.values() for v in vals],
            *PRIORITY_TO_PLAN.values(), *CONSULT_TEMPLATES, *DISTRACTOR_TURNS,
        ]:
            chars.update(s)
        vocab = {}
        for tok in SPECIAL_TOKENS:
            vocab[tok] = len(vocab)
        for ch in sorted(chars):
            if ch not in vocab:
                vocab[ch] = len(vocab)
        return cls(vocab)

    def encode(self, text):
        ids = []
        for part in self._split_re.split(text):
            if not part:
                continue
            if part in self.token_to_id:
                ids.append(self.token_to_id[part])
            else:
                for ch in part:
                    ids.append(self.token_to_id.get(ch, self.unk_token_id))
        return ids

    def decode(self, ids, skip_special_pad=True):
        out = []
        for i in ids:
            tok = self.id_to_token.get(i, '')
            if skip_special_pad and tok == '<|pad|>':
                continue
            out.append(tok)
        return ''.join(out)

@dataclass
class Episode:
    level: int
    prompt: str
    topic: str
    full_slots: dict
    known_slots: list
    memory_slots: list
    missing_slots: list
    expected_plan: str
    ideal_completion: str

def _fmt_known(slots, keys):
    return '、'.join(f'{k}は{slots[k]}' for k in keys)

def _fmt_kv(slots, keys):
    return '、'.join(f'{k}={slots[k]}' for k in keys)

def _build_summary(topic, slots, plan):
    return f"テーマ={topic} | 目的={slots['目的']} | 制約={slots['制約']} | 優先軸={slots['優先軸']} | 方針={plan}"

def generate_episode(level, rng, num_distractors=1):
    if level == 1:
        topic = rng.choice(TOPICS)
        full_slots = {k: rng.choice(SLOT_VALUES[k]) for k in SLOT_NAMES}
        missing = rng.choice(SLOT_NAMES)
        known = [k for k in SLOT_NAMES if k != missing]
        plan = PRIORITY_TO_PLAN[full_slots['優先軸']]
        prompt = rng.choice(CONSULT_TEMPLATES).format(topic=topic, known_text=_fmt_known(full_slots, known)) + '\n'
        ideal = (
            f'<think>テーマ: {topic}。既知: {_fmt_kv(full_slots, known)}。不足: {missing}</think>'
            f'<ask>{missing}</ask><user>{missing}: {full_slots[missing]}</user>'
            f'<think>整理完了。全前提が揃った。</think>'
            f'<summary>{_build_summary(topic, full_slots, plan)}</summary><|eos|>'
        )
        return Episode(1, prompt, topic, full_slots, known, [], [missing], plan, ideal)

    if level == 2:
        topic = rng.choice(TOPICS)
        full_slots = {k: rng.choice(SLOT_VALUES[k]) for k in SLOT_NAMES}
        known_slot = rng.choice(SLOT_NAMES)
        m1, m2 = [k for k in SLOT_NAMES if k != known_slot]
        plan = PRIORITY_TO_PLAN[full_slots['優先軸']]
        prompt = rng.choice(CONSULT_TEMPLATES).format(topic=topic, known_text=_fmt_known(full_slots, [known_slot])) + '\n'
        ideal = (
            f'<think>テーマ: {topic}。既知: {known_slot}={full_slots[known_slot]}。不足: {m1}、{m2}</think>'
            f'<ask>{m1}</ask><user>{m1}: {full_slots[m1]}</user>'
            f'<think>確認: {m1}={full_slots[m1]}。次は{m2}が必要。不足: {m2}</think>'
            f'<ask>{m2}</ask><user>{m2}: {full_slots[m2]}</user>'
            f'<think>整理完了。全前提が揃った。</think>'
            f'<summary>{_build_summary(topic, full_slots, plan)}</summary><|eos|>'
        )
        return Episode(2, prompt, topic, full_slots, [known_slot], [], [m1, m2], plan, ideal)

    topic1, topic2 = rng.sample(TOPICS, 2)
    mem_slot = rng.choice(['制約', '優先軸'])
    shared_val = rng.choice(SLOT_VALUES[mem_slot])
    slots1 = {k: rng.choice(SLOT_VALUES[k]) for k in SLOT_NAMES}
    slots1[mem_slot] = shared_val
    plan1 = PRIORITY_TO_PLAN[slots1['優先軸']]
    t1_known = [k for k in SLOT_NAMES if k != mem_slot]
    turn1 = (
        rng.choice(CONSULT_TEMPLATES).format(topic=topic1, known_text=_fmt_known(slots1, t1_known)) + '\n'
        f'<think>テーマ: {topic1}。既知: {_fmt_kv(slots1, t1_known)}。不足: {mem_slot}</think>'
        f'<ask>{mem_slot}</ask><user>{mem_slot}: {shared_val}</user>'
        f'<think>整理完了。全前提が揃った。</think>'
        f'<summary>{_build_summary(topic1, slots1, plan1)}</summary>\n'
    )
    distractors = '\n'.join(rng.choice(DISTRACTOR_TURNS) for _ in range(max(1, num_distractors))) + '\n'
    slots2 = {k: rng.choice(SLOT_VALUES[k]) for k in SLOT_NAMES}
    slots2[mem_slot] = shared_val
    rem = [k for k in SLOT_NAMES if k != mem_slot]
    known2 = rng.choice(rem)
    missing2 = [k for k in rem if k != known2][0]
    plan2 = PRIORITY_TO_PLAN[slots2['優先軸']]
    prompt = turn1 + distractors + rng.choice(CONSULT_TEMPLATES).format(topic=topic2, known_text=_fmt_known(slots2, [known2])) + '\n'
    ideal = (
        f'<think>テーマ: {topic2}。既知: {known2}={slots2[known2]}。記憶: {mem_slot}={shared_val}。不足: {missing2}</think>'
        f'<ask>{missing2}</ask><user>{missing2}: {slots2[missing2]}</user>'
        f'<think>整理完了。全前提が揃った。</think>'
        f'<summary>{_build_summary(topic2, slots2, plan2)}</summary><|eos|>'
    )
    return Episode(3, prompt, topic2, slots2, [known2], [mem_slot], [missing2], plan2, ideal)

tokenizer = ThinkingTokenizer.build_default()
(OUTPUT_DIR / 'vocab.json').write_text(json.dumps(tokenizer.token_to_id, ensure_ascii=False, indent=2), encoding='utf-8')
print('Vocab size:', tokenizer.vocab_size)

demo_rng = random.Random(0)
for lvl in (1, 2, 3):
    ep = generate_episode(lvl, demo_rng)
    print(f'\n--- Level {lvl} サンプル (tokens={len(tokenizer.encode(ep.prompt + ep.ideal_completion))}) ---')
    print(ep.prompt + ep.ideal_completion)

In [ ]:
# 4. Titans (MAC) モデル構築 & 疑似ユーザー対話ロールアウト・評価関数
if torch.cuda.is_available():
    device = torch.device('cuda')
    major, _ = torch.cuda.get_device_capability()
    amp_dtype = torch.bfloat16 if (major >= 8 and torch.cuda.is_bf16_supported()) else (torch.float16 if major >= 7 else torch.float32)
else:
    device, amp_dtype = torch.device('cpu'), torch.float32

use_amp = device.type == 'cuda' and amp_dtype in (torch.bfloat16, torch.float16)
use_scaler = device.type == 'cuda' and amp_dtype == torch.float16

model = MemoryAsContextTransformer(
    num_tokens=tokenizer.vocab_size,
    dim=cfg['dim'],
    depth=cfg['depth'],
    segment_len=cfg['segment_len'],
    num_persist_mem_tokens=cfg['num_persist_mem_tokens'],
    num_longterm_mem_tokens=cfg['num_longterm_mem_tokens'],
).to(device)
print(f'Titans (MAC) params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | device={device} | amp={amp_dtype}')

ASK_RE = re.compile(r'<ask>([^<>]+)</ask>')
SUMMARY_RE = re.compile(r'<summary>(.*?)</summary>', re.DOTALL)
THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL)

def _parse_summary(body):
    parsed = {}
    for seg in body.split('|'):
        seg = seg.strip()
        if '=' in seg:
            k, v = seg.split('=', 1)
            parsed[k.strip()] = v.strip()
    return parsed

def score_rollout(ep, comp_text, asks):
    has_think = bool(THINK_RE.search(comp_text))
    m = SUMMARY_RE.search(comp_text)
    format_ok = has_think and (m is not None)
    reward = 0.20 if format_ok else 0.0

    missing_set = set(ep.missing_slots)
    known_or_mem = set(ep.known_slots) | set(ep.memory_slots)
    seen, hit = set(), set()
    redundant = invalid = 0
    reasked_mem = False
    for q in asks:
        if q not in SLOT_NAMES:
            invalid += 1
            continue
        if q in seen:
            redundant += 1
            continue
        seen.add(q)
        if q in missing_set:
            hit.add(q)
        elif q in known_or_mem:
            redundant += 1
            if q in ep.memory_slots:
                reasked_mem = True

    if missing_set:
        reward += 0.50 * (len(hit) / len(missing_set))
    reward -= 0.25 * redundant + 0.20 * invalid
    if len(asks) > 3:
        reward -= 0.20
    ask_exact = (hit == missing_set) and (redundant == 0) and (invalid == 0)

    expected = {
        'テーマ': ep.topic, '目的': ep.full_slots['目的'],
        '制約': ep.full_slots['制約'], '優先軸': ep.full_slots['優先軸'], '方針': ep.expected_plan,
    }
    matched = 0
    if m is not None:
        parsed = _parse_summary(m.group(1))
        matched = sum(1 for k, v in expected.items() if parsed.get(k) == v)
    slot_acc = matched / len(expected)
    reward += 1.00 * slot_acc
    correct = (matched == len(expected)) and format_ok
    mem_reused = bool(ep.memory_slots) and correct and (not reasked_mem)
    if mem_reused:
        reward += 0.30
    return reward, correct, slot_acc, ask_exact, mem_reused, format_ok

@torch.no_grad()
def run_interactive_rollout(model, ep, max_new_tokens=MAX_NEW_TOKENS, temperature=GRPO_TEMP, top_p=0.92, greedy=False):
    model.eval()
    prompt_ids = tokenizer.encode(ep.prompt)
    seq = list(prompt_ids)
    action_mask = [0] * len(prompt_ids)
    asks = []
    for _ in range(max_new_tokens):
        if len(seq) >= cfg['seq_len']:
            break
        x = torch.tensor([seq], dtype=torch.long, device=device)
        logits = model(x)[:, -1, :]
        if greedy or temperature <= 1e-5:
            next_id = int(torch.argmax(logits, dim=-1).item())
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            remove = torch.cumsum(sorted_probs, dim=-1) > top_p
            remove[..., 1:] = remove[..., :-1].clone()
            remove[..., 0] = False
            sorted_probs[remove] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True).clamp_min(1e-8)
            next_id = int(sorted_idx.gather(-1, torch.multinomial(sorted_probs, 1)).item())
        seq.append(next_id)
        action_mask.append(1)
        if next_id == tokenizer.eos_token_id:
            break
        if next_id == tokenizer.ask_end_id and len(asks) < 3:
            matches = ASK_RE.findall(tokenizer.decode(seq[len(prompt_ids):]))
            if matches:
                q = matches[-1].strip()
                asks.append(q)
                reply = f'{q}: {ep.full_slots[q]}' if q in ep.full_slots else 'エラー: 該当なし'
                env_ids = tokenizer.encode(f'<user>{reply}</user>')
                if len(seq) + len(env_ids) < cfg['seq_len']:
                    seq.extend(env_ids)
                    action_mask.extend([0] * len(env_ids))
    comp_text = tokenizer.decode(seq[len(prompt_ids):])
    reward, correct, slot_acc, ask_exact, mem_reused, format_ok = score_rollout(ep, comp_text, asks)
    return dict(
        full_ids=seq, action_mask=action_mask, comp_text=comp_text, asks=asks,
        reward=reward, correct=correct, slot_acc=slot_acc, ask_exact=ask_exact,
        mem_reused=mem_reused, format_ok=format_ok,
    )

def evaluate_all(model, n_per_level=EVAL_PER_LEVEL, seed=9999):
    rng = random.Random(seed)
    results = {}
    for lvl in (1, 2, 3):
        c_cnt = f_cnt = a_cnt = m_cnt = 0
        s_sum = r_sum = 0.0
        shown = False
        for _ in range(n_per_level):
            ep = generate_episode(lvl, rng)
            ro = run_interactive_rollout(model, ep, greedy=True)
            c_cnt += int(ro['correct'])
            f_cnt += int(ro['format_ok'])
            a_cnt += int(ro['ask_exact'])
            m_cnt += int(ro['mem_reused'])
            s_sum += ro['slot_acc']
            r_sum += ro['reward']
            if not shown:
                shown = True
                print(f'  [L{lvl} 例] 不足={ep.missing_slots} | 記憶={ep.memory_slots} | 質問={ro["asks"]} | 判定={ro["correct"]}')
                print(f'         出力: {ro["comp_text"]}')
        results[f'level_{lvl}'] = {
            'accuracy': round(c_cnt / n_per_level, 3),
            'slot_accuracy': round(s_sum / n_per_level, 3),
            'ask_exact_rate': round(a_cnt / n_per_level, 3),
            'format_rate': round(f_cnt / n_per_level, 3),
            'avg_reward': round(r_sum / n_per_level, 3),
            **({'memory_reuse_rate': round(m_cnt / n_per_level, 3)} if lvl == 3 else {}),
        }
    return results

In [ ]:
# 5. Stage 1: カリキュラム SFT（基礎作り）
def tokenize_sft(ep):
    p_ids = tokenizer.encode(ep.prompt)
    c_ids = tokenizer.encode(ep.ideal_completion)
    ids = p_ids + c_ids
    lbls = [-100] * len(p_ids) + list(c_ids)
    u_s, u_e = tokenizer.token_to_id['<user>'], tokenizer.token_to_id['</user>']
    in_u = False
    for i in range(len(p_ids), len(lbls)):
        if ids[i] == u_s:
            in_u = True
            lbls[i] = -100
        elif in_u:
            lbls[i] = -100
            if ids[i] == u_e:
                in_u = False
    return ids[:cfg['seq_len']], lbls[:cfg['seq_len']]

def sample_level(progress, rng):
    r = rng.random()
    if progress < 0.35:
        return 1 if r < 0.75 else 2
    if progress < 0.70:
        return 1 if r < 0.25 else (2 if r < 0.75 else 3)
    return 1 if r < 0.15 else (2 if r < 0.55 else 3)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['sft_lr'], betas=(0.9, 0.95), weight_decay=0.05)
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler) if hasattr(torch.amp, 'GradScaler') else torch.cuda.amp.GradScaler(enabled=use_scaler)
sft_rng = random.Random(SEED)
warmup = max(1, min(60, SFT_STEPS // 10))

model.train()
t0 = time.time()
for step in range(1, SFT_STEPS + 1):
    progress = (step - 1) / max(1, SFT_STEPS)
    if step <= warmup:
        lr = cfg['sft_lr'] * step / warmup
    else:
        p = (step - warmup) / max(1, SFT_STEPS - warmup)
        lr = cfg['sft_lr'] * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * p)))
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    for _ in range(cfg['sft_grad_accum']):
        batch = [tokenize_sft(generate_episode(sample_level(progress, sft_rng), sft_rng)) for _ in range(cfg['sft_batch_size'])]
        mlen = max(len(a) for a, _ in batch)
        x = torch.full((len(batch), mlen), tokenizer.pad_token_id, dtype=torch.long)
        y = torch.full((len(batch), mlen), -100, dtype=torch.long)
        for i, (a, b) in enumerate(batch):
            x[i, :len(a)] = torch.tensor(a)
            y[i, :len(b)] = torch.tensor(b)
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with (torch.amp.autocast(device_type=device.type, dtype=amp_dtype) if use_amp else nullcontext()):
            logits = model(x[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y[:, 1:].reshape(-1), ignore_index=-100)
            scaled = loss / cfg['sft_grad_accum']
        if use_scaler:
            scaler.scale(scaled).backward()
        else:
            scaled.backward()
        accum_loss += float(loss.item()) / cfg['sft_grad_accum']

    if use_scaler:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    if step % 50 == 0 or step == 1 or step == SFT_STEPS:
        print(f'[SFT] step {step:4d}/{SFT_STEPS} | loss {accum_loss:.4f} | lr {lr:.2e} | {time.time() - t0:.1f}s')

torch.save({'stage': 'sft', 'model_state_dict': model.state_dict()}, OUTPUT_DIR / 'stage1_sft_latest.pt')
if device.type == 'cuda':
    torch.cuda.empty_cache()

print('\n=== Stage 1 (SFT) 完了後の評価 ===')
sft_metrics = evaluate_all(model)
print(json.dumps(sft_metrics, ensure_ascii=False, indent=2))

In [ ]:
# 6. Stage 2: GRPO（Level 2 & 3 重点・段階報酬つき強化学習）
def seq_logprobs(m, full_ids, action_mask):
    x = torch.tensor([full_ids[:-1]], dtype=torch.long, device=device)
    tgt = torch.tensor([full_ids[1:]], dtype=torch.long, device=device)
    mask = torch.tensor([action_mask[1:]], dtype=torch.float32, device=device)
    with (torch.amp.autocast(device_type=device.type, dtype=amp_dtype) if use_amp else nullcontext()):
        lp = F.log_softmax(m(x).float(), dim=-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return lp.squeeze(0), mask.squeeze(0)

ref_model = copy.deepcopy(model).to(device).eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

rl_opt = torch.optim.AdamW(model.parameters(), lr=cfg['grpo_lr'], betas=(0.9, 0.95), weight_decay=0.01)
rl_rng = random.Random(SEED + 1000)
t0 = time.time()
ema_r = ema_acc = 0.0
updated_steps = 0

for step in range(1, GRPO_STEPS + 1):
    r_lvl = rl_rng.random()
    lvl = 1 if r_lvl < 0.10 else (2 if r_lvl < 0.55 else 3)
    n_dist = rl_rng.choice([1, 2]) if lvl == 3 else 1
    ep = generate_episode(lvl, rl_rng, num_distractors=n_dist)

    rollouts = [run_interactive_rollout(model, ep, temperature=GRPO_TEMP, greedy=False) for _ in range(GROUP_SIZE)]
    rewards = [ro['reward'] for ro in rollouts]
    mean_r = sum(rewards) / GROUP_SIZE
    std_r = math.sqrt(sum((r - mean_r) ** 2 for r in rewards) / GROUP_SIZE)
    acc_g = sum(1.0 for ro in rollouts if ro['correct']) / GROUP_SIZE
    ema_r = mean_r if step == 1 else 0.9 * ema_r + 0.1 * mean_r
    ema_acc = acc_g if step == 1 else 0.9 * ema_acc + 0.1 * acc_g

    if std_r < 1e-6:
        if step % 10 == 0 or step == 1 or step == GRPO_STEPS:
            print(f'[GRPO] step {step:3d}/{GRPO_STEPS} | L{lvl} | r={mean_r:.3f} (ema {ema_r:.3f}) | acc={acc_g:.2f} | upd={updated_steps}/{step} | skip(zero-var)')
        continue

    updated_steps += 1
    advs = [(r - mean_r) / (std_r + 1e-6) for r in rewards]
    model.train()
    rl_opt.zero_grad(set_to_none=True)
    total_kl = 0.0
    for ro, adv in zip(rollouts, advs):
        if sum(ro['action_mask'][1:]) == 0:
            continue
        pol_lp, mask = seq_logprobs(model, ro['full_ids'], ro['action_mask'])
        with torch.no_grad():
            ref_lp, _ = seq_logprobs(ref_model, ro['full_ids'], ro['action_mask'])
        denom = mask.sum().clamp_min(1.0)
        log_ratio = (ref_lp - pol_lp).clamp(-10.0, 10.0)
        kl = ((torch.exp(log_ratio) - log_ratio - 1.0) * mask).sum() / denom
        pg = -adv * (pol_lp * mask).sum() / denom
        ((pg + KL_BETA * kl) / GROUP_SIZE).backward()
        total_kl += float(kl.item()) / GROUP_SIZE

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    rl_opt.step()
    if device.type == 'cuda' and step % 10 == 0:
        torch.cuda.empty_cache()
    if step % 10 == 0 or step == 1 or step == GRPO_STEPS:
        print(f'[GRPO] step {step:3d}/{GRPO_STEPS} | L{lvl} | r={mean_r:.3f} (ema {ema_r:.3f}) | acc={acc_g:.2f} | upd={updated_steps}/{step} | kl={total_kl:.4f} | {time.time() - t0:.1f}s')

torch.save({'stage': 'grpo', 'model_state_dict': model.state_dict()}, OUTPUT_DIR / 'stage2_grpo_latest.pt')
del ref_model
if device.type == 'cuda':
    torch.cuda.empty_cache()

print(f'\n=== Stage 2 (GRPO) 完了後の評価 (更新ステップ: {updated_steps}/{GRPO_STEPS}) ===')
grpo_metrics = evaluate_all(model)
print(json.dumps(grpo_metrics, ensure_ascii=False, indent=2))
(OUTPUT_DIR / 'metrics.json').write_text(
    json.dumps({'after_sft': sft_metrics, 'after_grpo': grpo_metrics, 'grpo_updated_steps': updated_steps}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

In [ ]:
# 7. 自由テストセル（任意の相談で逆質問→思考整理の動きを確認）
custom_ep = Episode(
    level=2,
    prompt='相談: アプリ開発について思考を整理したい。目的はスキル習得。方針を決めたい。\n',
    topic='アプリ開発',
    full_slots={'目的': 'スキル習得', '制約': '平日30分', '優先軸': '継続しやすさ'},
    known_slots=['目的'],
    memory_slots=[],
    missing_slots=['制約', '優先軸'],
    expected_plan=PRIORITY_TO_PLAN['継続しやすさ'],
    ideal_completion='',
)
ro = run_interactive_rollout(model, custom_ep, greedy=True)
print('相談:', custom_ep.prompt.strip())
print('逆質問履歴:', ro['asks'])
print('完全一致:', ro['correct'], '| スロット一致率:', ro['slot_acc'], '| 報酬:', round(ro['reward'], 3))
print('思考＋整理結果:\n', ro['comp_text'])